
# 🔐 Nemo Safe Synthesizer Tutorial: The Basics

#### What you'll learn

In this notebook, we'll explore the fundamentals of the NeMo Safe Synthesizer: optional PII replacement (strongly recommended), training on a sample dataset, generating synthetic data, and evaluating quality and privacy.

This microservice supports numeric, categorical, and text fields within the training data and generates realistic synthetic data that mirrors the structure of your data. A full run takes ~20 minutes with and without PII replacement on an A-100; an H-100 is faster.

### 🖥️ Prerequisites

This notebook is intended to run on a **GPU**. We recommend an **H-100**; minimum **A-100**.




### ⚡ Colab Setup

Run the cell below to install Nemo Safe Synthesizer (engine and CUDA 12.8) and kagglehub for the sample dataset. Package access uses the NVIDIA Artifactory index.

In [ ]:
%%capture
!uv pip install nemo-safe-synthesizer[engine,cu128] --extra-index-url "https://urm.nvidia.com/artifactory/api/pypi/nv-shared-pypi-local/simple"
!uv pip install kagglehub



### 🔑 Set the NIM API key and configure column classification:

Setting `NIM_API_KEY` is optional but strongly recommended.

NeMo Safe Synthesizer uses an LLM‑based column classifier to automatically infer column types and improve PII detection accuracy. To enable this feature, you must set both `NIM_ENDPOINT_URL` and `NIM_API_KEY`. You can obtain an API key from [build.nvidia.com](https://build.nvidia.com/settings/api-keys)

The current default model for column classification is `qwen/qwen2.5-coder-32b-instruct`. You can override it by setting the `NIM_MODEL_ID` environment variable to any OpenAI‑compatible endpoint.

In [ ]:
import os
import webbrowser
import getpass

# Set the NIM endpoint URL
os.environ["NIM_ENDPOINT_URL"] = "https://integrate.api.nvidia.com/v1"
print("NIM_ENDPOINT_URL is set.")

# setting NIM_API_KEY is optional but strongly recommanded for PII replacement.
if "NIM_API_KEY" not in os.environ:
    os.environ["NIM_API_KEY"] = getpass.getpass("Paste NIM API key (or press Enter to skip): ")
if os.environ.get("NIM_API_KEY"):
    print("NIM_API_KEY is set")
else:
    print(
        "NIM_API_KEY is not set, hence PII replacement is skipped."
        "We strongly recommend setting a key."
    )

### 📥 Load and preview sample dataset:

Load a tabular dataset—in this example, the Women’s E‑Commerce Clothing Reviews dataset from Kaggle—and preview the first few rows. NeMo Safe Synthesizer will use this DataFrame as its training data.

This dataset includes text, categorical, and numeric fields, making it a good demonstration of the model’s ability to handle multiple data types.

In [ ]:
import pandas as pd
import kagglehub 

# Download the dataset
path = kagglehub.dataset_download("nicapotato/womens-ecommerce-clothing-reviews")
df = pd.read_csv(f"{path}/Womens Clothing E-Commerce Reviews.csv", index_col=0)

print(f"The size of the dataframe is: {df.shape}")
df.head()





###  ⚙️  create and run Safe Synthesizer job:

Create the Safe Synthesizer builder and attach your DataFrame. You can enable PII replacement by adding `with_replace_pii()` to the builder, which requires the `NIM_API_KEY` environment variable. Model training and generation are enabled by adding `synthesize().resolve()` to the builder.

The `with_` methods configure the builder and do not execute the model. Refer to the [reference docs](https://aire.gitlab-master-pages.nvidia.com/microservices/nmp/latest/nemo-microservices/latest/safe-synthesizer/about/reference.html) for the full list of configuration options.

Run the pipeline with `run()`, which performs data processing, PII replacement, training, generation, and evaluation in a single call. Results are available on `builder.results`.

In [ ]:
from nemo_safe_synthesizer.sdk.library_builder import SafeSynthesizer

builder = SafeSynthesizer().with_data_source(df).with_replace_pii().synthesize().resolve()
builder.run()
results = builder.results

### 📤 Retrieve synthetic data:

Inspect the generated synthetic data including row count and preview of the first rows.

In [ ]:
synth = results.synthetic_data
print(f"Number of synthetic rows: {len(synth)}")
synth.head()

### 🛡️ Review evaluation report:

The pipeline computes both quality and privacy metrics. The summary includes timing information and overall scores, while the full evaluation report is rendered as an HTML document.

In [ ]:
import json 

print("Summary (timing and scores):")
print(json.dumps(results.summary.model_dump(), indent=2))

In [ ]:
# Download the full HTML evaluation report
if results.evaluation_report_html:
    report_path = "evaluation_report.html"
    with open(report_path, "w") as f:
        f.write(results.evaluation_report_html)
    print(f"The HTML evaluation report is saved in {report_path}.")

### ➡️ Next Steps

Now that you've completed your first Safe Synthesizer job, explore more advanced features:

### Advanced Tutorials

- [Differential Privacy Tutorial](https://aire.gitlab-master-pages.nvidia.com/microservices/nmp/latest/nemo-microservices/latest/safe-synthesizer/tutorials/differential-privacy.html) - Apply mathematical privacy guarantees

- [PII Replacement Tutorial](https://aire.gitlab-master-pages.nvidia.com/microservices/nmp/latest/nemo-microservices/latest/safe-synthesizer/tutorials/pii-replacement.html) - Advanced PII detection and replacement


### Try These Next

1. **Customize PII replacement**: Configure specific entity types and replacement strategies
2. **Enable differential privacy**: Add formal privacy guarantees with epsilon and delta parameters
3. **Tune generation parameters**: Adjust temperature and sampling for better synthetic data
4. **Use your own data**: Replace the sample dataset with your sensitive data
